# Native quantum programming, part II

Work through each exercise and compare with its executable FlagQuantum solution.

Set up the local FlagQuantum session.

In [ ]:
from ast import FunctionDef, parse
from inspect import getsource

import flagquantum as fq
from examples.synthesis.tutorials.basics.quantum_programming.qmod_tutorial_part2.overview import quantum, reference, workflow


def definition(name: str) -> str:
    source = getsource(quantum).splitlines()
    for node in parse(getsource(quantum)).body:
        if isinstance(node, FunctionDef) and node.name == name and node.end_lineno is not None:
            start = min((item.lineno for item in node.decorator_list), default=node.lineno)
            return chr(10).join(source[start - 1 : node.end_lineno])
    raise ValueError(name)

7A. Encode the product of two two-bit numbers as a phase.

In [ ]:
product, result, checked = workflow.run_native(
    'product_phase', reference.phase_state(), ('x', 'y'), max_qubits=4, angles=True
)
print(checked.vector.tolist())
print(product.report['resources'])

7B. Mark the two solutions of 3a + b = 9 without changing their probabilities.

In [ ]:
oracle, result, marked = workflow.run_native(
    'solution_oracle', reference.phase_state(oracle=True), ('a', 'b'), max_qubits=4
)
print(marked.vector.tolist())

8. Compute 2x + 1 in a local register. Flip the output if the value exceeds three, then uncompute.

In [ ]:
comparison, result, local = workflow.run_native(
    'local_comparison', reference.comparison_state(), ('x', 'res'),
    max_qubits=7, max_workspace=4,
)
print(workflow.read_counts(comparison, result, ('x', 'res'), seed=10))
print('Scratch leakage:', local.workspace_probability)

9A. Add a three-bit quantum number to a four-bit accumulator initialized to two.

In [ ]:
addition, result, simple_add = workflow.run_native(
    'fourier_add', reference.addition_state(controlled=False), ('x', 'y'),
    max_qubits=7, angles=True,
)
print(workflow.read_counts(addition, result, ('x', 'y'), seed=11))

9B. Control only the phase block. The Fourier transform and its inverse surround both branches.

In [ ]:
controlled, result, guarded_add = workflow.run_native(
    'controlled_fourier_add', reference.addition_state(controlled=True),
    ('ctrl', 'x', 'y'), max_qubits=8,
)
print(workflow.read_counts(controlled, result, ('ctrl', 'x', 'y'), seed=12))

10A. Apply a two-step, second-order Suzuki–Trotter formula at time three on four qubits.

In [ ]:
evolution, result, evolved = workflow.run_native(
    'pauli_evolution', reference.trotter_state(), ('qarr',), max_qubits=4
)
print(evolution.report['resources'])
print(workflow.read_counts(evolution, result, ('qarr',), seed=13))

10B. Observe X(0) + X(1) on the prepared plus pair.

In [ ]:
plus, plus_error = workflow.observe_plus()
print(plus, plus_error)

11A. Pass a one-qubit operation to a reusable Python function and apply it to three qubits.

In [ ]:
callback_counts, callback_error = workflow.read_callback()
print(callback_counts, callback_error)

11B. Use a controlled RZ phase callback and four fractional phase bits.

In [ ]:
phase, phase_state = workflow.read_qpe()
print(dict(phase), phase_state.error)

12A. Build one symbolic template for both sampling bindings.

In [ ]:
template = quantum.parameterized_ansatz()
print(fq.__name__, template.parameter_names)

12B. Define a third binding for the observable experiment on the same template.

In [ ]:
print(getsource(workflow.observe_binding))

Solution 7A. Decompose the bilinear phase by bit significance using four controlled phases.

In [ ]:
print(definition('product_phase'))
print('Product phase maximum amplitude error:', checked.error)

Solution 7B. Use the predicate to apply a phase of pi.

In [ ]:
print(definition('mark'))
print('Predicate maximum amplitude error:', marked.error)

Solution 8. Use within/apply to give the local register a verified inverse.

In [ ]:
print(definition('local_comparison'))
assert local.workspace_probability == 0

Solution 9A. The inverse transform translates phase into modular addition.

In [ ]:
print(definition('fourier_add'))
print('Addition maximum amplitude error:', simple_add.error)

Solution 9B. Superpose the control and check both branches including their relative phase.

In [ ]:
print(definition('controlled_fourier_add'))
print('Controlled addition maximum amplitude error:', guarded_add.error)

Solution 10A. The independent reference uses dense Pauli exponentials for the same product formula.

In [ ]:
print(definition('pauli_evolution'))
print('Trotter maximum amplitude error:', evolved.error)

Solution 10B. Request the observable expectation on the native prepared circuit.

In [ ]:
print(getsource(workflow.observe_plus))
print('X(0) + X(1):', plus)

Solution 11A. Use a typed Python callable at circuit construction.

In [ ]:
print(getsource(quantum.apply_to_all))
print('Finite three-qubit samples:', callback_counts)

Solution 11B. A conditional phase polynomial exactly expresses CRZ(pi), including the other eigenphase.

In [ ]:
print(definition('estimate_phase'))
print('Decoded fractional phase:', dict(phase))

Solution 12A. Bind and sample both settings without rebuilding the symbolic template.

In [ ]:
samples = workflow.read_bindings(template)
print('Finite samples from the two bindings:', samples)
print('Unchanged symbolic template:', template.parameter_names)

Solution 12B. Bind the same template for a third observable measurement.

In [ ]:
energy, energy_error = workflow.observe_binding(template)
print('Bound expectation:', energy, 'maximum amplitude error:', energy_error)